# Mini-project — One region, one process: what satellites see, and what a reanalysis does with it

**Prerequisites**: TP1, TP2 (Copernicus Marine downloads, TP2a) and TP3.

Until now, the questions were chosen for you. This afternoon, you study **one ocean process** from start to end: the **upwelling of the Gulf of Panama**, driven by the wind that crosses the isthmus. You look at it with **satellite observations of the sea surface** (wind, temperature, currents, chlorophyll), then you check what the **GLORYS12 reanalysis** does with it. **Each pair takes a different year**: tomorrow (Friday), you present your year in **5 slides**, and together we compare the years.

## How this mini-project works

- Work **alone or in pairs**, **one year per pair** (the trainers will help you share the years).
- The notebook is a **guided path**, the same for every year, with `# TODO` cells to complete. Most of the code you need is already in TP1, TP2 and TP3.
- The **✅ Check** boxes give the numbers for the worked example, **February 2023**. For your year the numbers will be different — that is the point! — but the steps and the kind of result are the same. Not sure your code is right? Run it once with `YEAR = 2023` and compare.
- Stuck on a `# TODO`? Open the **💡 Hint** box, then, as a last resort, the **🔑 Solution** box below the cell.
- Every figure is **saved automatically** in the folder `figures_panama_<your year>/`: these files go directly into your slides.
- Make your own copy first: right-click this notebook → *Duplicate*, rename it `04_miniproject_EN_mine.ipynb`, and **keep it in this folder** (it uses the helper functions of `../Session02/tp2_utils.py`).

| Time (Thursday) | What you do |
|---|---|
| 14:00 – 14:20 | **1.** The study area, your year, your question |
| 14:20 – 14:45 | **2.** Download the satellite observations |
| 14:45 – 15:40 | **3.** The process seen by satellites |
| 15:40 – 16:00 | Break |
| 16:00 – 16:40 | **4.** The same process in the GLORYS reanalysis |
| 16:40 – 17:00 | **5.** Your year in one line, **6.** from your figures to 5 slides |
| **Friday** | **5 slides, 5 minutes**, then questions |

## Contents
1. [Setup — imports and helper functions](#0-setup)
2. [The study area, your year, your question](#1-choose)
3. [Download the satellite observations](#2-download)
4. **[The process seen by satellites](#3-obs)**
   - [3.1 The mean picture of the month](#3-1)
   - [3.2 Forcing and response: time series](#3-2)
   - [3.3 Zoom on an event](#3-3)
5. **[The same process in the GLORYS reanalysis](#4-glorys)**
   - [4.1 Download GLORYS](#4-1)
   - [4.2 SST: maps and bias](#4-2)
   - [4.3 Currents](#4-3)
   - [4.4 Time series and scores](#4-4)
6. [Your year in one line](#5-summary)
7. [From your figures to 5 slides](#6-slides)

---
<a id="0-setup"></a>
## 0. Setup — imports and helper functions

Run this cell as is. Besides `download()`, `show_dataset()` and `base_map()` (from TP2, in `../Session02/tp2_utils.py`), it defines four small helpers used all along the notebook:

| Helper | What it does |
|---|---|
| `box_mean(da, box)` | mean of `da` over a box `[lon_min, lon_max, lat_min, lat_max]` → one value per day |
| `draw_box(ax, box)` | draws a box on a map |
| `arrows(ax, u, v, step)` | vectors (wind or currents) on a map, one arrow every `step` grid points |
| `save(fig, name)` | saves the figure in `figures_<case>/<name>.png`, for your slides |

In [ ]:
import sys
import calendar
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cartopy.crs as ccrs
import cmocean

# Data already on Kabre (read-only): the trainers' backup copies of the downloads are there
OBS_DIR = "/data/gcambon/COMMONDATA/DATASETS_PSF/"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"

# Where your downloads go (created next to this notebook)
DATA_DIR = Path("data_tp4")
DATA_DIR.mkdir(exist_ok=True)

# Helper functions of TP2: download(), show_dataset(), base_map()
sys.path.insert(0, str(Path("../Session02").resolve()))
from tp2_utils import setup, download, show_dataset, base_map


def box_mean(da, box):
    """Mean of da over box = [lon_min, lon_max, lat_min, lat_max] -> one value per time step."""
    sub = da.sel(longitude=slice(box[0], box[1]), latitude=slice(box[2], box[3]))
    return sub.mean(["latitude", "longitude"])


def draw_box(ax, box, color="magenta", **kwargs):
    """Draw box = [lon_min, lon_max, lat_min, lat_max] on a map."""
    lon0, lon1, lat0, lat1 = box
    ax.plot([lon0, lon1, lon1, lon0, lon0], [lat0, lat0, lat1, lat1, lat0], color=color,
            linewidth=2, transform=ccrs.PlateCarree(), zorder=5, **kwargs)


def arrows(ax, u, v, step=1, key=None, units="m/s", color="k"):
    """Vectors (u, v) on a map, one arrow every `step` grid points; `key` = length of the legend arrow."""
    s = slice(None, None, step)
    q = ax.quiver(u["longitude"].values[s], u["latitude"].values[s], u.values[s, s], v.values[s, s],
                  color=color, transform=ccrs.PlateCarree(), zorder=4)
    if key:
        ax.quiverkey(q, 0.90, 0.04, key, f"{key} {units}", labelpos="N", coordinates="axes",
                     fontproperties={"size": 9}, zorder=6)
    return q


def save(fig, name):
    """Save a figure in figures_<case>/<name>.png, ready for your slides."""
    FIG_DIR.mkdir(exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    print(f"💾 {FIG_DIR / name}.png")


print("✓ setup done")

**✅ Check**: `✓ setup done`. A `ModuleNotFoundError: No module named 'tp2_utils'` means that your notebook is not in the `Session04` folder: move it back there.

---
<a id="1-choose"></a>
## 1. The study area, your year, your question

**The process: a gap-wind upwelling.** In winter, the trade winds from the Caribbean cross Central America through the **low gaps of the mountains** — the Isthmus of Panama, where the canal is, is the lowest one — and blow as a **wind jet** over the Pacific. The jet pushes the surface water away and mixes it: cold, nutrient-rich water comes up from below. **What satellites see**: a **cold tongue** of SST under the jet, more **chlorophyll** (the nutrients feed the phytoplankton), strong **currents**, and a cooling a **few days after each wind burst**. February is the peak of the season.

**The Gulf of Panama is an example of study area.** The same notebook works for any region where a process is visible from space — another wind jet (Papagayo in Costa Rica, Tehuantepec in Mexico), a coastal upwelling, a river plume: you only change the boxes and the month. Here everybody works on the same region, so that the trainers can help everyone, and so that on Friday we can **compare the years**.

**Each pair takes one year** (always February). The upwelling is not the same every year: the wind, the depth of the cold water and the large-scale ocean change, for example with **El Niño and La Niña** (TP2b). The phase of ENSO in winter, from the NOAA ONI index:

| Year | ENSO in winter (Dec–Feb) |
|---|---|
| 2010 | El Niño (moderate) |
| 2011 | La Niña (strong) |
| 2013 | neutral |
| 2016 | El Niño (very strong) |
| 2018 | La Niña (weak) |
| 2019 | El Niño (weak) |
| 2021 | La Niña (moderate) |
| 2023 ⭐ | La Niña (ending) — the worked example of the ✅ Check boxes |
| 2024 | El Niño (strong) |
| 2025 | La Niña (weak) — a surprising year: see for yourself! |

Other years from 2007 to 2025 also work: ask a trainer.

The notebook uses **three boxes**:
- `BOX`: the **study area**, downloaded and mapped;
- `CORE`: where the **cold water spreads**, for the time series;
- `WIND_BOX`: in the **wind jet**, just south of the isthmus (the forcing).

**🎯 Your turn**: set your year (`YEAR = ...`), run the cell, and look at the map.

In [ ]:
# Study area: the Gulf of Panama (an example: the same steps work for any region)
REGION = "panama"
BOX = [-82, -77.5, 4, 9.5]            # study area [lon_min, lon_max, lat_min, lat_max]
CORE = [-80.3, -79, 6, 7.8]           # where the cold water spreads (time series)
WIND_BOX = [-80, -79.2, 7.6, 8.4]     # in the wind jet, just south of the isthmus (forcing)

YEAR = 2023                           # ← your year (one per pair)

MONTH = f"{YEAR}-02"                  # February: the peak of the upwelling season
START, END = f"{MONTH}-01", f"{MONTH}-{calendar.monthrange(YEAR, 2)[1]:02d}"
CASE = f"{REGION}_{YEAR}"
FIG_DIR = Path(f"figures_{CASE}")
setup(box=BOX, obs_dir=OBS_DIR, data_dir=DATA_DIR)
assert 2007 <= YEAR <= 2025, "The wind product starts in 2007: choose a year between 2007 and 2025"
print(f"Gulf of Panama, {START} to {END}")

fig, ax = plt.subplots(figsize=(9, 6.5), subplot_kw={"projection": ccrs.PlateCarree()})
base_map(ax, f"Gulf of Panama — study box (black), core box (magenta), wind box (dashed)",
         extent=[BOX[0] - 6, BOX[1] + 6, BOX[2] - 4, BOX[3] + 4])
draw_box(ax, BOX, "k")
draw_box(ax, CORE)
draw_box(ax, WIND_BOX, "k", linestyle="--")
save(fig, "01_region")
plt.show()

**✅ Check**: `Gulf of Panama, 2023-02-01 to 2023-02-28` (with your year; 29 days in a leap year: 2012, 2016, 2020, 2024). The study box covers the **Gulf of Panama** and the Pacific south of it, down to 4°N. The wind box (dashed) is inside the gulf, just south of the isthmus, where the trade winds coming from the Caribbean cross the lowest point of Central America; the core box (magenta) is south of it, where the cold water spreads.

**📝 Your question** — double-click on this cell and write, in one or two sentences, **what you want to find out** about your year (it is your slide 1). For example: *"In February 2016, during a strong El Niño, was the Panama upwelling as cold and as green as usual, and does the GLORYS reanalysis reproduce it?"*

> *Our question:* …

---
<a id="2-download"></a>
## 2. Download the satellite observations

Four products from **Copernicus Marine**, the same tool as in TP2a (`download()` skips a file that is already there). One month, the study box only: a few MB per file.

| What | `dataset_id` | `variables` | Resolution |
|---|---|---|---|
| SST (OSTIA, L4) | `METOFFICE-GLO-SST-L4-REP-OBS-SST` | `["analysed_sst"]` | 0.05°, daily, **kelvin** |
| Wind (scatterometers, L4) | `cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H` | `["eastward_wind", "northward_wind"]` | 0.125°, **hourly**, m/s at 10 m |
| Surface currents (GlobCurrent) | `cmems_obs-mob_glo_phy-cur_my_0.25deg_P1D-m` | `["uo", "vo"]` | 0.25°, daily, depths 0 and 15 m |
| Chlorophyll (GlobColour, gap-free L4) | `cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D` | `["CHL"]` | 4 km, daily, mg m⁻³ |

💬 **No satellite measures currents directly.** The "observed" currents combine the **geostrophic** current, computed from the sea level measured by altimeters, and the **Ekman** current, computed from the wind. In the same way, the L4 wind fills the gaps between satellite passes with a weather model, and the L4 SST and chlorophyll are **interpolated** under the clouds (Course 1). Keep it in mind when you write "observed" on a slide.

**🎯 Your turn**: the SST download is complete; complete the three others with the table above.

In [ ]:
AREA = dict(minimum_longitude=BOX[0], maximum_longitude=BOX[1],
            minimum_latitude=BOX[2], maximum_latitude=BOX[3])

SST_FILE = download(f"{REGION}_sst_{MONTH}.nc",
                    dataset_id="METOFFICE-GLO-SST-L4-REP-OBS-SST", variables=["analysed_sst"],
                    start_datetime=START, end_datetime=END, **AREA)

WIND_FILE = download(f"{REGION}_wind_{MONTH}.nc",
                     dataset_id=...,          # TODO
                     variables=...,           # TODO (2 variables)
                     start_datetime=f"{START}T00:00:00", end_datetime=f"{END}T23:00:00", **AREA)   # hourly: give the hours

CUR_FILE = download(f"{REGION}_currents_{MONTH}.nc",
                    dataset_id=...,           # TODO
                    variables=...,            # TODO
                    minimum_depth=..., maximum_depth=...,   # TODO: the surface only
                    start_datetime=START, end_datetime=END, **AREA)

CHL_FILE = download(f"{REGION}_chl_{MONTH}.nc",
                    dataset_id=...,           # TODO
                    variables=...,            # TODO
                    start_datetime=START, end_datetime=END, **AREA)


<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Copy the `dataset_id` and the `variables` of each line of the table: `variables` is always a **list**, e.g. `["uo", "vo"]`.
- The currents have two depths, 0 and 15 m: `minimum_depth=0, maximum_depth=0` keeps the surface only.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
 dataset_id="cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H",
 variables=["eastward_wind", "northward_wind"],
dataset_id="cmems_obs-mob_glo_phy-cur_my_0.25deg_P1D-m",
variables=["uo", "vo"],
minimum_depth=0, maximum_depth=0,
dataset_id="cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D",
variables=["CHL"],
```

</details>

**✅ Check**: four lines `✓ data_tp4/...`, and no error. For February 2023, about **0.6 MB** for the SST, **4 MB** for the wind (it is hourly), **0.1 MB** for the currents and **1.6 MB** for the chlorophyll. `Complete the TODOs first` means that a `...` is left in a cell.

Now open the files. The SST is converted to °C right away, and the currents keep their surface level only.

In [ ]:
sst = xr.open_dataset(SST_FILE)["analysed_sst"] - 273.15       # kelvin -> °C
wind = xr.open_dataset(WIND_FILE)                              # hourly
cur = xr.open_dataset(CUR_FILE).isel(depth=0)                  # surface currents
chl = xr.open_dataset(CHL_FILE)["CHL"]

for name, da in [("SST", sst), ("wind", wind["eastward_wind"]), ("currents", cur["uo"]), ("chl", chl)]:
    print(f"{name:9}", dict(da.sizes), "| latitude increasing:", bool(da["latitude"][0] < da["latitude"][-1]))

**✅ Check** (February 2023): every product has `time`, `latitude`, `longitude`, with **latitudes increasing** (so `slice(lat_min, lat_max)` works, TP1):

| Product | Sizes |
|---|---|
| SST | `time: 28, latitude: 110, longitude: 90` |
| wind | `time: 672, latitude: 44, longitude: 36` |
| currents | `time: 28, latitude: 22, longitude: 18` |
| chl | `time: 28, latitude: 132, longitude: 108` |

Four different grids, from 4 km to 0.25°! And the wind has **672 time steps**: 28 days × 24 hours.

**🎯 Your turn**: bring the wind to **daily** values, like the other products, then compute the wind **speed**.

In [ ]:
# TODO: daily means of the two wind components (hint: .resample(), TP2b)
u10 = ...
v10 = ...
# TODO: the daily wind speed (m/s)
speed = ...
print(dict(speed.sizes), f"| strongest daily wind: {float(speed.max()):.1f} m/s")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `wind["eastward_wind"].resample(time="1D").mean()` gives one value per day.
- The speed is the length of the vector: `np.sqrt(u10**2 + v10**2)`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
u10 = wind["eastward_wind"].resample(time="1D").mean()
v10 = wind["northward_wind"].resample(time="1D").mean()
speed = np.sqrt(u10**2 + v10**2)
```

</details>

**✅ Check** (February 2023): `{'time': 28, 'latitude': 44, 'longitude': 36}`, and a strongest daily wind of **11.0 m/s** (somewhere in the jet, on one day).

---
<a id="3-obs"></a>
# 3. The process seen by satellites

<a id="3-1"></a>
## 3.1 The mean picture of the month

**🎯 Your turn**: compute the **monthly means** (mean over `time`), then look at two maps: the SST with the wind, and the chlorophyll with the currents. The chlorophyll uses a **logarithmic** colour scale, as in TP2c: it varies from 0.05 to more than 5 mg m⁻³.

In [ ]:
# TODO: mean over "time" of sst, u10, v10, chl, and of the currents cur["uo"], cur["vo"]
sst_m = ...
u10_m, v10_m = ..., ...
chl_m = ...
cu_m, cv_m = ..., ...

fig, axs = plt.subplots(1, 2, figsize=(16, 5.5), subplot_kw={"projection": ccrs.PlateCarree()})
sst_m.plot(ax=axs[0], cmap=cmocean.cm.thermal, cbar_kwargs={"label": "SST (°C)", "shrink": 0.85})
arrows(axs[0], u10_m, v10_m, step=4, key=10)
base_map(axs[0], f"Satellite SST and wind (arrows) — mean of {MONTH}")
chl_m.plot(ax=axs[1], cmap=cmocean.cm.algae, norm=LogNorm(0.05, 5),
           cbar_kwargs={"label": "Chlorophyll (mg m⁻³, log scale)", "shrink": 0.85})
arrows(axs[1], cu_m, cv_m, step=2, key=0.5)
base_map(axs[1], f"Satellite chlorophyll and surface currents (arrows) — mean of {MONTH}")
for ax in axs:
    draw_box(ax, CORE)
    draw_box(ax, WIND_BOX, "k", linestyle="--")
save(fig, "02_obs_mean_maps")
plt.show()
print(f"SST: {float(sst_m.min()):.1f} to {float(sst_m.max()):.1f} °C | strongest mean wind: {float(np.hypot(u10_m, v10_m).max()):.1f} m/s"
      f" | strongest mean current: {float(np.hypot(cu_m, cv_m).max()):.2f} m/s")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- The same for all: `.mean("time")`, e.g. `sst_m = sst.mean("time")`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
sst_m = sst.mean("time")
u10_m, v10_m = u10.mean("time"), v10.mean("time")
chl_m = chl.mean("time")
cu_m, cv_m = cur["uo"].mean("time"), cur["vo"].mean("time")
```

</details>

**✅ Check** (February 2023): SST from **23.7 to 29.2 °C**; strongest mean wind **8.6 m/s**; strongest mean current **1.23 m/s**. On the maps:
- **left**: the wind blows from north to south across the isthmus, then fans out to the south-south-west over the gulf. Under it lies a **cold tongue**: ≈ 25.0 °C on average in the core box, **2 to 3 °C colder** than the water to the west (≈ 28.1 °C) and to the east (≈ 27.1 °C);
- **right**: the chlorophyll is **3 times higher in the cold tongue** (≈ 1.6 mg m⁻³ in the core box) than on both sides (≈ 0.5 mg m⁻³): the upwelled water brings **nutrients** to the surface. The currents carry the cold, green water out of the gulf to the south-west, at up to ≈ 1.2 m/s.

**For your case**: is there a cold or a fresh/green signal, and where is it compared with the wind and the coast? Is it inside your core box? If not, **move the core box** (in section 1) and run again: that is part of the job.

---
<a id="3-2"></a>
## 3.2 Forcing and response: time series

A map shows **where**; a time series shows **when**, and **how fast** the ocean responds. Here: the wind speed in the wind box (the **forcing**), and the SST and the chlorophyll in the core box (the **response**), day by day.

**🎯 Your turn**: compute the three daily series with `box_mean()` (defined in the setup).

In [ ]:
# TODO: daily series with box_mean(data, box)
wind_ts = ...          # wind speed in WIND_BOX
sst_ts = ...           # SST in CORE
chl_ts = ...           # chlorophyll in CORE

fig, axs = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axs[0].plot(wind_ts["time"], wind_ts, color="k", marker=".", label="wind speed (wind box)")
axs[0].set_ylabel("Wind speed (m/s)")
ax_t = axs[0].twinx()
ax_t.plot(sst_ts["time"], sst_ts, color="tab:red", marker=".", label="SST (core box)")
ax_t.set_ylabel("SST (°C)", color="tab:red")
axs[0].set_title(f"{CASE} — forcing (wind) and response (SST, chlorophyll), {MONTH}")
axs[0].legend(loc="upper left")
ax_t.legend(loc="upper right")
axs[1].plot(chl_ts["time"], chl_ts, color="tab:green", marker=".")
axs[1].set_ylabel("Chlorophyll (mg m⁻³)\n(core box)")
for ax in axs:
    ax.grid(alpha=0.3)
save(fig, "03_time_series")
plt.show()
print(f"wind: {float(wind_ts.min()):.1f} to {float(wind_ts.max()):.1f} m/s (strongest on {str(wind_ts.idxmax().values)[:10]})")
print(f"SST : {float(sst_ts.min()):.2f} to {float(sst_ts.max()):.2f} °C (coldest on {str(sst_ts.idxmin().values)[:10]})")
print(f"chl : {float(chl_ts.min()):.2f} to {float(chl_ts.max()):.2f} mg m⁻³ (highest on {str(chl_ts.idxmax().values)[:10]})")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `box_mean(speed, WIND_BOX)`: the wind **speed** in the wind box. Then the same with `sst` and `chl` in `CORE`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
wind_ts = box_mean(speed, WIND_BOX)
sst_ts = box_mean(sst, CORE)
chl_ts = box_mean(chl, CORE)
```

</details>

**✅ Check** (February 2023): wind from **4.7 to 9.1 m/s** (strongest on 4 February), SST from **22.12 to 26.75 °C** (coldest on **28 February**), chlorophyll from **0.46 to 4.00 mg m⁻³** (highest on 22 February). Look at the shapes rather than the numbers:
- the SST **goes down all month**: February–March is the peak of the upwelling season in the Gulf of Panama. But it goes down **in steps**;
- **after the wind bursts of 12–13 and 18–19 February, the SST drops quickly**: from 25.9 °C (12 February) to 25.0 °C (15 February), then from 25.2 °C (18 February) to 23.4 °C (21 February). Between the bursts it stays flat, or even warms a little (16–17 February);
- the chlorophyll goes from ≈ 0.5 mg m⁻³ at the start to a **bloom of 4 mg m⁻³ on 22 February**, a few days after the 18–19 February burst: the phytoplankton needs a few days to use the nutrients brought up by the cold water.

**For your year**: is the SST going down during the month? Do you see the same "wind burst → cooling" steps, and a chlorophyll bloom after them? Note the dates: they go on your slide 4.

---
<a id="3-3"></a>
## 3.3 Zoom on an event

A monthly mean smooths everything. Now look at **one event**: how much did the SST **change in the two days after the strongest wind**? A map of SST two days later minus SST on the day of the strongest wind: negative values = the sea has **cooled**.

In [ ]:
day = wind_ts.idxmax()                                   # date of the strongest wind in the wind box
later = sst["time"].sel(time=day + np.timedelta64(2, "D"), method="nearest")   # two days later (or the last day)
# TODO: SST change = SST two days later (later) minus SST on the day of the strongest wind (day)
sst_change = ...

fig, ax = plt.subplots(figsize=(10, 6.5), subplot_kw={"projection": ccrs.PlateCarree()})
sst_change.plot(ax=ax, cmap="RdBu_r", vmin=-2, vmax=2, cbar_kwargs={"label": "SST change (°C)", "shrink": 0.85})
arrows(ax, u10.sel(time=day), v10.sel(time=day), step=4, key=10)
base_map(ax, f"SST change from {str(day.values)[:10]} to {str(later.values)[:10]} — wind (arrows) on {str(day.values)[:10]}")
draw_box(ax, CORE)
draw_box(ax, WIND_BOX, "k", linestyle="--")
save(fig, "04_event")
plt.show()
print(f"Strongest wind on {str(day.values)[:10]}; SST change in the core box two days later: {float(box_mean(sst_change, CORE)):+.2f} °C")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `sst.sel(time=later)` and `sst.sel(time=day)` are two maps: subtract them (later minus day).

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
sst_change = sst.sel(time=later) - sst.sel(time=day)
```

</details>

**✅ Check** (February 2023): strongest wind on **2023-02-04**; two days later, the core box has **cooled by ≈ 0.4 °C**, with a cooling (blue) under the jet, from the gulf down to 5°N, and some warming west of 80.5°W. The biggest drops of the month come later (3.2): the strongest wind is not always the strongest response — it also depends on how long the wind blows and on how close the cold water is to the surface. **Try another day**: replace the first line with `day = wind_ts["time"].sel(time="2023-02-18")` and run again.

💬 **Is it the wind's fault?** One event is not a proof. The time series (3.2) shows the **same sequence several times**: that is a much stronger argument for your slides.

---
<a id="4-glorys"></a>
# 4. The same process in the GLORYS reanalysis

GLORYS12 (TP2a) is an ocean model at 1/12° (≈ 9 km), run over 1993 to today, and **corrected every week with observations** (altimetry, SST, Argo profiles…): a **reanalysis** (Course 2). Is the process there, at the right place, with the right strength and timing?

<a id="4-1"></a>
## 4.1 Download GLORYS

**🎯 Your turn**: in TP2a you downloaded **monthly** GLORYS (`..._P1M-m`). Here you need **daily** values (`..._P1D-m`) to follow the events, at the **surface** only: temperature and the two components of the current.

In [ ]:
GLORYS_FILE = download(f"{REGION}_glorys_{MONTH}.nc",
                       dataset_id=...,        # TODO: GLORYS12, daily
                       variables=...,         # TODO: temperature and the two current components
                       minimum_depth=0, maximum_depth=1,               # the first level only (0.49 m)
                       start_datetime=START, end_datetime=END, **AREA)

gl = xr.open_dataset(GLORYS_FILE).isel(depth=0)
print(gl)

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- The monthly id was `cmems_mod_glo_phy_my_0.083deg_P1M-m`: the daily one ends with `P1D-m`.
- `variables=["thetao", "uo", "vo"]`, as in TP2a.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
variables=["thetao", "uo", "vo"],
```

</details>

**✅ Check** (February 2023): about **0.6 MB**; `time: 28, latitude: 67, longitude: 55` (1/12°: finer than the satellite currents, coarser than the OSTIA SST), variables `thetao` (°C, not kelvin!), `uo`, `vo`.

---
<a id="4-2"></a>
## 4.2 SST: maps and bias

To subtract two maps, they must be on the **same grid**. OSTIA is at 0.05°, GLORYS at 1/12°: **interpolate** OSTIA onto the GLORYS grid with `.interp()`, then compute the **bias** = GLORYS − OSTIA (positive = GLORYS too warm).

**🎯 Your turn**: complete the two lines.

In [ ]:
gl_sst = gl["thetao"]
# TODO: OSTIA SST interpolated on the GLORYS grid (hint: .interp(latitude=..., longitude=...))
sst_on_gl = ...
# TODO: mean bias map over the month, GLORYS minus OSTIA
bias_map = ...

vmin, vmax = float(sst_m.min()), float(sst_m.max())          # the same colour scale for both
fig, axs = plt.subplots(1, 3, figsize=(20, 5), subplot_kw={"projection": ccrs.PlateCarree()})
sst_m.plot(ax=axs[0], cmap=cmocean.cm.thermal, vmin=vmin, vmax=vmax, cbar_kwargs={"label": "SST (°C)", "shrink": 0.8})
base_map(axs[0], f"Satellite (OSTIA) — {MONTH}")
gl_sst.mean("time").plot(ax=axs[1], cmap=cmocean.cm.thermal, vmin=vmin, vmax=vmax, cbar_kwargs={"label": "SST (°C)", "shrink": 0.8})
base_map(axs[1], f"Reanalysis (GLORYS12) — {MONTH}")
bias_map.plot(ax=axs[2], cmap="RdBu_r", vmin=-1.5, vmax=1.5, cbar_kwargs={"label": "GLORYS − OSTIA (°C)", "shrink": 0.8})
base_map(axs[2], "Bias of GLORYS (red: too warm, blue: too cold)")
for ax in axs:
    draw_box(ax, CORE)
save(fig, "05_sst_obs_vs_glorys")
plt.show()
print(f"Bias map: {float(bias_map.min()):+.2f} to {float(bias_map.max()):+.2f} °C, mean {float(bias_map.mean()):+.2f} °C")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `sst.interp(latitude=gl_sst["latitude"], longitude=gl_sst["longitude"])`: the OSTIA values at the GLORYS points.
- The bias: `(gl_sst - sst_on_gl).mean("time")` — GLORYS first, so that red = GLORYS too warm.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
sst_on_gl = sst.interp(latitude=gl_sst["latitude"], longitude=gl_sst["longitude"])
bias_map = (gl_sst - sst_on_gl).mean("time")
```

</details>

**✅ Check** (February 2023): bias from **−1.1 to +1.7 °C**, mean **+0.36 °C**. GLORYS has the **cold tongue at the right place** — not surprising, since GLORYS **assimilates satellite SST** (so it is not an independent comparison!) — but it is **too warm in the gulf and in the core box** (red, ≈ +1 °C): the upwelling is **too weak** in GLORYS.

---
<a id="4-3"></a>
## 4.3 Currents

The satellite currents (0.25°, from altimetry + wind) against the GLORYS currents (1/12°, from the model physics). No interpolation this time: just compare the two maps, with the **speed** in colour.

In [ ]:
# TODO: mean current speed (m/s) of the satellite (cu_m, cv_m) and of GLORYS (gl["uo"], gl["vo"], mean over time)
gu_m, gv_m = ..., ...
speed_sat = ...
speed_gl = ...

fig, axs = plt.subplots(1, 2, figsize=(16, 5.5), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, spd, u, v, step, title in [(axs[0], speed_sat, cu_m, cv_m, 1, "Satellite (GlobCurrent, 0.25°)"),
                                   (axs[1], speed_gl, gu_m, gv_m, 3, "Reanalysis (GLORYS12, 1/12°)")]:
    spd.plot(ax=ax, cmap=cmocean.cm.speed, vmin=0, vmax=1, cbar_kwargs={"label": "Current speed (m/s)", "shrink": 0.85})
    arrows(ax, u, v, step=step, key=0.5)
    base_map(ax, f"{title} — mean surface current, {MONTH}")
    draw_box(ax, CORE)
save(fig, "06_currents_obs_vs_glorys")
plt.show()
print(f"Mean current speed in the core box: satellite {float(box_mean(speed_sat, CORE)):.2f} m/s, GLORYS {float(box_mean(speed_gl, CORE)):.2f} m/s")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `gl["uo"].mean("time")`, then the speed as for the wind: `np.sqrt(u**2 + v**2)`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
gu_m, gv_m = gl["uo"].mean("time"), gl["vo"].mean("time")
speed_sat = np.sqrt(cu_m**2 + cv_m**2)
speed_gl = np.sqrt(gu_m**2 + gv_m**2)
```

</details>

**✅ Check** (February 2023): in the core box, **0.71 m/s** (satellite) and **0.62 m/s** (GLORYS); strongest mean currents ≈ 1.2 and 1.4 m/s. The two maps agree **less** than the SST maps: both show a strong current leaving the gulf to the south-west, but GLORYS organises it in a large **loop** around the gulf, while the satellite shows a broad flow to the south-west at 5–6°N. Which one is "right"? Neither is a direct measurement: discuss it on your slide 5.

---
<a id="4-4"></a>
## 4.4 Time series and scores

Last step: does GLORYS follow the **events** of 3.2? Compare the daily SST in the core box, and compute three classic **scores**:
- the **bias** (mean of GLORYS − OSTIA): a systematic error;
- the **RMSE** (root mean square error): the typical size of the error, day by day;
- the **correlation**: do the two curves go up and down together?

**🎯 Your turn**: complete the scores.

In [ ]:
gl_ts = box_mean(gl_sst, CORE)
obs_ts = box_mean(sst_on_gl, CORE)            # OSTIA on the GLORYS grid: the same points
diff = gl_ts - obs_ts
# TODO: the three scores
bias = ...
rmse = ...
corr = ...          # hint: xr.corr(a, b)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(obs_ts["time"], obs_ts, color="tab:red", marker=".", label="satellite (OSTIA)")
ax.plot(gl_ts["time"], gl_ts, color="tab:blue", marker=".", label="reanalysis (GLORYS12)")
ax.set_ylabel("SST in the core box (°C)")
ax.set_title(f"{CASE} — bias {bias:+.2f} °C, RMSE {rmse:.2f} °C, correlation {corr:.2f}")
ax.legend()
ax.grid(alpha=0.3)
save(fig, "07_sst_series_obs_vs_glorys")
plt.show()
print(f"bias {bias:+.2f} °C | RMSE {rmse:.2f} °C | correlation {corr:.2f}")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- bias = mean of `diff`; RMSE = square root of the mean of `diff**2`; correlation = `xr.corr(gl_ts, obs_ts)`.
- Use `float(...)` to get plain numbers.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
bias = float(diff.mean())
rmse = float(np.sqrt((diff**2).mean()))
corr = float(xr.corr(gl_ts, obs_ts))
```

</details>

**✅ Check** (February 2023): bias **+0.49 °C**, RMSE **0.58 °C**, correlation **0.98**. GLORYS follows **every step** of the cooling (correlation 0.98), but it is **too warm, and more and more**: the same as OSTIA at the start of the month, ≈ 1 °C warmer at the end (23.1 °C against 22.1 °C on 28 February). Even with the satellite SST assimilated, the upwelling is not strong enough in GLORYS. A good point for your slide 5!

**🔁 Cross-check** — the same bias, computed in two independent ways, must give the same number (a habit of scientific rigour, as in TP1b):

In [ ]:
bias_from_map = float(box_mean(bias_map, CORE))      # method 2: average of the bias MAP over the core box
print(f"bias from the time series: {bias:+.3f} °C | bias from the map: {bias_from_map:+.3f} °C")

**✅ Check**: the two numbers are equal, or almost (**+0.490** and **+0.486 °C** for February 2023). Averaging over time then space, or over space then time, gives the same mean when every point has a value every day; a difference in the third decimal comes from a few coastal points that are missing on some days. A larger difference means that the two calculations do not use the same points (land, box, period).

---
<a id="5-summary"></a>
## 5. Your year in one line

Every pair has studied a different year. To compare them on Friday, put **your numbers in one line** and copy it into the **shared table** (the trainers give you the link, or write it on the board).

In [ ]:
summary = {
    "year": YEAR,
    "mean wind (m/s)": round(float(wind_ts.mean()), 1),
    "SST core mean (°C)": round(float(sst_ts.mean()), 2),
    "SST core min (°C)": round(float(sst_ts.min()), 2),
    "chl core mean (mg m-3)": round(float(chl_ts.mean()), 2),
    "chl core max (mg m-3)": round(float(chl_ts.max()), 2),
    "GLORYS bias (°C)": round(bias, 2),
    "RMSE (°C)": round(rmse, 2),
    "correlation": round(corr, 2),
}
print(" | ".join(f"{k}: {v}" for k, v in summary.items()))

**✅ Check** (February 2023): `year: 2023 | mean wind (m/s): 6.8 | SST core mean (°C): 25.01 | SST core min (°C): 22.12 | chl core mean (mg m-3): 1.59 | chl core max (mg m-3): 4.0 | GLORYS bias (°C): 0.49 | RMSE (°C): 0.58 | correlation: 0.98`

On Friday, the table of all the years is the starting point of the discussion: which years had the **coldest** and the **greenest** upwelling? Is it related to the **wind**, or to **El Niño / La Niña**? Does **GLORYS** do better in some years than in others?

---
<a id="6-slides"></a>
## 6. From your figures to 5 slides

Your figures are in `figures_panama_<your year>/`: download them (right-click → *Download* in the file browser on the left) and build **5 slides** for a **5-minute** presentation on Friday.

| Slide | Content | Figures |
|---|---|---|
| 1 | **The question**: the process, the region, the month, and your question (section 1) | `01_region.png` |
| 2 | **The data**: a table of the products (what, resolution, observation or model) | — |
| 3 | **The process seen by satellites**: what, where | `02_obs_mean_maps.png` (or `02b_obs_salinity.png`), `04_event.png` |
| 4 | **Forcing and response**: when, how fast | `03_time_series.png` |
| 5 | **Satellites vs GLORYS**, then **your conclusion** (and how your year compares with the others) and **one limit** of your study | `05_sst_obs_vs_glorys.png`, `07_sst_series_obs_vs_glorys.png` |

**Tips for good slides**
- **One message per slide**, written as a sentence in the title: *"The SST drops about 1 day after each wind burst"* rather than *"Time series"*.
- Units, dates and colour-bar labels on every figure; say which data are **observations** and which are **model**.
- Give your numbers with their meaning: *"bias +0.1 °C, small compared with the 2 °C of the cold tongue"*.
- **One honest limit**: only one month; the satellite products are interpolated (L4); GLORYS assimilates the same satellites, so the comparison is not independent…

**✅ Final checklist**
- [ ] a question in one sentence (section 1)
- [ ] the figures in `figures_panama_<year>/`, and your line in the shared table (section 5)
- [ ] three numbers on slide 5: bias, RMSE, correlation
- [ ] one limit of the study